In [0]:
-- Using all codes (2 years)
create or replace temporary view mpsii_treatment_table as
SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          --  WHERE PROCEDURE_CODE IN ('J1743')
)
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31';

In [0]:
create or replace temporary view mpsii_diagnosis_table as 
with mpsii_1dx_specified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_specified as (
  select *
  from mpsii_1dx_specified
  where patient_id in (select a.patient_id from mpsii_1dx_specified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
mpsii_1dx_unspecified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_unspecified as (
  select *
  from mpsii_1dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_1dx_unspecified as a group by a.patient_id having count(distinct a.fill_date) >= 2) 
),
mpsii_2dx_specified_tx as (
  select *
  from mpsii_treatment_table where patient_id in (select distinct a.patient_id from mpsii_2dx_specified as a)
),
incremental_patient as (
  select distinct patient_id
  from mpsii_2dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_treatment_table as a where a.code in ('54092070001','540920700','J1743'))
  and patient_id not in (select distinct b.patient_id from mpsii_2dx_specified_tx as b)
),
all_dx_patients_claims as (
  select * from mpsii_2dx_specified
  union
  select * from mpsii_1dx_unspecified where patient_id in (select distinct a.patient_id from incremental_patient as a) 
)
select *
from all_dx_patients_claims

In [0]:
/* =============================================================================
   PURPOSE
   -----------------------------------------------------------------------------
   This script assigns a single “Primary HCP” NPI per eligible patient by:
   1) Building Dx claims (5-year window) using aligned NPI attribution rules
   2) Building Tx claims (5-year window) using corrected NPI attribution rules
   3) Defining eligibility cohorts (Specified + Incremental) using Dx frequency
      and Tx presence in a 2-year window
   4) Combining Dx + Tx claims for eligible patients to form an HCP visit history
   5) Ranking candidate HCPs per patient using a 4-tier method:
        Tier 1: Specialty priority
        Tier 2: Total visit count (Dx + Tx combined)
        Tier 3: Most recent visit date
        Tier 4: NPI tiebreaker (ascending)
   6) Enriching the selected Primary HCP with an affiliated HCO NPI from the
      reference_file mapping

   TIME WINDOWS
   -----------------------------------------------------------------------------
   - Diagnosis window (5 years): 2020-08-01 to 2025-07-31
   - Treatment window (5 years): 2020-08-01 to 2025-07-31
   - Treatment eligibility window (2 years): 2023-08-01 to 2025-07-31

   NPI ATTRIBUTION RULES (aligned with GTM file)
   -----------------------------------------------------------------------------
   - Medical events with NDC:       COALESCE(RENDERING_NPI, REFERRING_NPI)
   - Medical events with procedure: RENDERING_NPI only
   - Pharmacy events:              PRESCRIBER_NPI

   NOTES
   -----------------------------------------------------------------------------
   - This script creates multiple TEMP views used downstream.
   - Step labels below reflect the actual order of object creation.
   ============================================================================= */


/* =============================================================================
   STEP 1: DIAGNOSIS CLAIMS (5yr)
   - Builds a unified Dx claims view across:
     a) Medical events (Dx identified from DIAGNOSIS_CODES; NPI = COALESCE)
     b) Pharmacy events (Dx identified from DIAGNOSIS_CODE; NPI = PRESCRIBER)
   - Output fields standardized to: PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical Events - Dx (COALESCE(RENDERING_NPI, REFERRING_NPI))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

UNION

-- Pharmacy Events - Dx (PRESCRIBER_NPI)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31';


/* =============================================================================
   STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
   - Builds a unified Tx claims view across:
     a) Medical events with NDC codes (NPI = COALESCE)
     b) Medical events with procedure codes (NPI = RENDERING only)
     c) Pharmacy events with NDC codes (NPI = PRESCRIBER)
   - Includes a CODE field to support eligibility logic (e.g., Elaprase)
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical Events - NDC codes (COALESCE(RENDERING_NPI, REFERRING_NPI))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

UNION

-- Medical Events - Procedure codes (RENDERING_NPI only)
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

UNION

-- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31';


/* =============================================================================
   STEP 3: TREATMENT CLAIMS (2yr subset for eligibility)
   - Restricts Tx claims to the 2-year eligibility window
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    CLAIM_TYPE,
    CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31';


/* =============================================================================
   STEP 4: PATIENT ELIGIBILITY
   -----------------------------------------------------------------------------
   Defines two mutually exclusive eligible cohorts:
   A) Specified:
      - ≥ 2 distinct E761 Dx dates in 5yr window
      - AND any Tx in 2yr window
   B) Incremental:
      - ≥ 2 distinct E763 Dx dates in 5yr window
      - AND Elaprase Tx in 2yr window (subset of Tx codes)
      - AND NOT already Specified
   Output: eligible_patients = specified ∪ incremental
   ============================================================================= */

-- 4A) Specified prerequisite: 2+ E761 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    -- Medical Dx dates for E761
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

    UNION

    -- Pharmacy Dx dates for E761 (paid only)
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4B) Specified patients: 2+ E761 Dx + any Tx in 2yr
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t
    ON e.PATIENT_ID = t.PATIENT_ID;

-- 4C) Incremental prerequisite: 2+ E763 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    -- Medical Dx dates for E763
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

    UNION

    -- Pharmacy Dx dates for E763 (paid only)
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4D) Elaprase Tx in 2yr (subset used for incremental eligibility)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

-- 4E) Incremental patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t
    ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- 4F) All eligible patients = specified ∪ incremental
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


/* =============================================================================
   STEP 5: COMBINE Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
   - Builds the visit history that will drive primary HCP ranking
   - Note: Visit counting uses DISTINCT FILL_DATE per patient/NPI (Dx + Tx combined)
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- Dx claims (5yr) for eligible patients
SELECT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    CLAIM_TYPE
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

-- Tx claims (5yr) for eligible patients
SELECT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    CLAIM_TYPE
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);


/* =============================================================================
   STEP 6: PRIMARY HCP ASSIGNMENT (4-tier ranking)
   -----------------------------------------------------------------------------
   6A) Build per-patient/per-NPI metrics:
       - Specialty bucket + specialty priority
       - Total visits (Dx + Tx combined) for ranking
       - Dx-only and Tx-only counts (reference)
       - Most recent visit date for ranking
   6B) Rank NPIs per patient using:
       Tier 1: specialty priority (ASC)
       Tier 2: total visits (DESC)
       Tier 3: most recent visit (DESC)
       Tier 4: NPI (ASC)
   6C) Select top-ranked HCP as PRIMARY_HCP_NPI
   6D) Enrich primary HCP with affiliated HCO NPI from reference_file
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH

/* ---------------------------------------------------------------------------
   6A: HCP metrics per patient/NPI (Dx + Tx combined)
   --------------------------------------------------------------------------- */
hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty Classification (human-readable grouping)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Specialty Priority (Tier 1: lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: Visit count uses Dx + Tx combined (distinct visit dates)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference counts (not used for ranking tiers)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: Most recent visit
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),

/* ---------------------------------------------------------------------------
   6B: Rank candidate HCPs per patient using 4-tier ordering
   --------------------------------------------------------------------------- */
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,  -- Tier 1: Specialty
                NO_OF_VISITS DESC,       -- Tier 2: Total visits (Dx + Tx)
                MOST_RECENT_VISIT DESC,  -- Tier 3: Most recent visit
                NPI ASC                  -- Tier 4: deterministic tiebreaker
        ) AS HCP_RANK
    FROM hcp_metrics
),

/* ---------------------------------------------------------------------------
   6C: Select the top-ranked HCP as the Primary HCP for each patient
   --------------------------------------------------------------------------- */
primary_hcp AS (
    SELECT
        PATIENT_ID,
        NPI AS PRIMARY_HCP_NPI
    FROM ranked_hcps
    WHERE HCP_RANK = 1
),

/* ---------------------------------------------------------------------------
   6D: Enrich Primary HCP with affiliated HCO NPI
   - Uses reference_file mapping (HCP NPI → HCO NPI)
   --------------------------------------------------------------------------- */
enriching_with_hco AS (
    SELECT
        a.*,
        b.hco_npi
    FROM primary_hcp AS a
    LEFT JOIN cmpa_insights_internal_schema.reference_file AS b
        ON a.primary_hcp_npi = b.hcp_npi
)

/* ---------------------------------------------------------------------------
   6E: Final output for primary_hcp view
   --------------------------------------------------------------------------- */
SELECT *
FROM enriching_with_hco;


/* -- OPTIONAL: Persist the view into a physical table (commented out)
   CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
   SELECT * FROM primary_hcp_assignment;
*/


In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi as
with base_table as (
  select * from com_edp_prd.com_raw.vod_npi where entity_type = 'HCO'
),
active_id as (
  select distinct vid__v, id as active_id
from base_table
where is_displayed = true
),
id_and_active_id as (
  select distinct a.vid__v, a.id as secondary_npi, b.active_id as primary_npi
  from base_table as a
  left join active_id as b
  on a.vid__v = b.vid__v
),
final_output as (
  select vid__v, secondary_npi, primary_npi
from id_and_active_id
where secondary_npi != '-'
order by primary_npi, secondary_npi
),
final_output_with_name as (
  select a.*, b.corporate_name__v as primary_name
  from final_output as a
  left join com_raw.vod_hco as b on a.primary_npi = b.npi_num__v
)
select * from final_output_with_name

In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds a final HCP–HCO reference view by:
   1) Starting from a base reference file
   2) Keeping records that already have an HCO NPI
   3) Deriving missing HCO affiliations using VOD (Salesforce) via HCP→HCO
   4) Backfilling remaining missing affiliations using Komodo
   5) Applying manual NPI remapping (Jess mapping logic)
   6) Mapping secondary NPIs to primary NPIs (Julie mapping logic)
   7) Flagging target HCOs based on an approved NPI list (plus primary-mapped NPIs)
   8) Enriching HCO ZIPs using VOD first, then Komodo as fallback
   9) Reassigning territory & region based on ZIP (HCO ZIP preferred, else HCP ZIP)
  10) Producing a final, clean HCP–HCO reference output

   NOTES
   ----------------------------------------------------------------------------
   - Documentation below is aligned to the *actual* flow of CTEs in the query.
   - No SQL logic has been changed; only comments/step labels were corrected.
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW reference_file_v1 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Load base reference file
   --------------------------------------------------------------------------- */
base_table AS (
    SELECT *
    FROM cmpa_insights_internal_schema.base_reference_file
),

/* ---------------------------------------------------------------------------
   STEP 1: Split records by presence of HCO NPI in the base file
   - v1: Records that already have an HCO NPI
   - v2: Records missing HCO NPI but having HCP NPI (eligible for affiliation search)
   --------------------------------------------------------------------------- */
vod_search_base_table_v1 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi != '-'
),

vod_search_base_table_v2 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi = '-'
      AND hcp_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 2: Fetch HCP VID from VOD using HCP NPI (needed to traverse HCP→HCO)
   --------------------------------------------------------------------------- */
hcp_vid AS (
    SELECT
        a.*,
        b.vid__v AS hcp_vid
    FROM vod_search_base_table_v2 a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* ---------------------------------------------------------------------------
   STEP 3: Pull active HCP–HCO affiliations from VOD and rank by recency
   - Filters: active parent HCO status + relationship type
   - Ranking: most recent modified/status update timestamp wins
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v1 AS (
    SELECT
        a.hcp_npi,
        c.vid__v AS hco_vid,
        c.corporate_name__v AS hco_name,
        c.npi_num__v AS hco_npi,
        b.modified_date__v,
        b.status_update_time__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

/* ---------------------------------------------------------------------------
   STEP 4: Select most recent active affiliation per HCP from VOD (rn=1)
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v2 AS (
    SELECT
        hcp_npi,
        hco_npi,
        hco_name
    FROM ranked_vod_affiliations_v1
    WHERE rn = 1
      AND hco_npi IS NOT NULL
),

/* ---------------------------------------------------------------------------
   STEP 5: Apply VOD-derived HCOs to base records missing HCO NPI (from Step 1 v2)
   --------------------------------------------------------------------------- */
vod_affiliations_implementation AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(b.hco_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM vod_search_base_table_v2 a
    LEFT JOIN ranked_vod_affiliations_v2 b
        ON a.hcp_npi = b.hcp_npi
),

/* ---------------------------------------------------------------------------
   STEP 6: Combine:
   - original records that already had HCO NPI (Step 1 v1)
   - records enriched with VOD-derived HCO NPI/name (Step 5)
   --------------------------------------------------------------------------- */
union_after_vod_check AS (
    SELECT * FROM vod_search_base_table_v1
    UNION
    SELECT * FROM vod_affiliations_implementation
),

/* ---------------------------------------------------------------------------
   STEP 7: Split into records still missing HCO affiliations vs. those with HCO
   --------------------------------------------------------------------------- */
hcp_with_no_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi = '-'
),

hcp_with_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 8: Backfill missing HCOs using Komodo affiliations
   - For INDIVIDUAL HCP NPI: use kom_providers.hco_primary_npi
   - Join to ORGANIZATION record to get HCO name
   --------------------------------------------------------------------------- */
pulling_affiliations_from_komodo AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_primary_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(c.organization_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM hcp_with_no_affiliations a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.com_raw.kom_providers c
        ON b.hco_primary_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 9: Combine all affiliations (VOD + Komodo)
   --------------------------------------------------------------------------- */
finalizing_affiliations AS (
    SELECT * EXCEPT(hco_target)
    FROM (
        SELECT * FROM hcp_with_affiliations
        UNION
        SELECT * FROM pulling_affiliations_from_komodo
    )
),

/* ---------------------------------------------------------------------------
   STEP 10: Manual NPI remapping table (Jess mapping logic)
   - Remaps specific HCO NPIs/names to corrected "mapped" NPIs/names
   --------------------------------------------------------------------------- */
npi_mapping AS (
    SELECT current_npi, current_name, mapped_npi, mapped_name
    FROM (
        VALUES
        ('1144211301','Atrium Health Wake Forest Baptist Medical Center','1295789907','Atrium Health'),
        ('1184779332','Childrens Healthcare Of Atlanta Scottish Rite Hospital','1235339227','Emory University Hospital'),
        ('1851458038','Dr Patrick Leavey MD Office','1235582925','UT Health'),
        ('1225259039','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1831318856','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1356496772','Kaiser Permanente Fontana Medical Center','1013062769','Kaiser Permanente San Diego Medical Center'),
        ('1003947599','Univ. Pediatric Associates Inc.','1144266024','Indiana University Health'),
        ('1205822236','Yale Medicine','1013924182','Yale-New Haven Hospital')
    ) t (current_npi, current_name, mapped_npi, mapped_name)
),

/* ---------------------------------------------------------------------------
   STEP 11: Apply manual NPI + name overrides (Jess mapping)
   --------------------------------------------------------------------------- */
jess_file_implementation AS (
    SELECT
        * EXCEPT (a.hco_npi, a.hco_name),
        COALESCE(b.mapped_npi, a.hco_npi) AS hco_npi,
        COALESCE(b.mapped_name, a.hco_name) AS hco_name
    FROM finalizing_affiliations a
    LEFT JOIN npi_mapping b
        ON a.hco_npi = b.current_npi
),

/* ---------------------------------------------------------------------------
   STEP 12: Map secondary NPIs to primary NPIs (Julie mapping logic)
   - Produces active HCO NPI/name fields:
       hco_npi_active, hco_name_active
   - Flags if the original HCO NPI appeared as a secondary in Julie mapping table
   --------------------------------------------------------------------------- */
julie_file_mapping AS (
    SELECT
        a.*,
        COALESCE(b.primary_npi, a.hco_npi) AS hco_npi_active,
        CASE WHEN b.secondary_npi IS NOT NULL THEN 1 ELSE 0 END
            AS hco_npi_present_julies_file_flag,
        CASE
            WHEN b.primary_npi IS NOT NULL THEN b.primary_name
            ELSE a.hco_name
        END AS hco_name_active
    FROM jess_file_implementation a
    LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi b
        ON a.hco_npi = b.secondary_npi
),

/* ---------------------------------------------------------------------------
   STEP 13: Set HCO target flag
   - Target if:
     a) hco_npi_active is in the approved target list OR
     b) hco_npi_active is the mapped primary NPI for a target-listed secondary NPI
   --------------------------------------------------------------------------- */
updating_hco_target_flag AS (
    SELECT
        a.*,
        CASE
            WHEN (a.hco_npi_active IN ('1932280666','1912939703','1891765178','1861439952','1851458038',
                '1831318856','1760480503','1760476659','1750482022','1750458485',
                '1700128592','1689747552','1679973364','1669683512','1669462420',
                '1669429577','1659877280','1649347469','1649261462','1639370059',
                '1609824010','1598784555','1578693321','1568596765','1548212988',
                '1477643690','1477549756','1467525790','1447423959','1437365186',
                '1396882205','1376544320','1366556227','1366515488','1356496772',
                '1346297843','1336495910','1336245828','1326092404','1295789907',
                '1285832634','1285647933','1285174649','1275694184','1275564098',
                '1265694442','1235582925','1235339227','1235234535','1235214834',
                '1235148594','1225259039','1225249865','1215921457','1205935012',
                '1205822236','1194787218','1184779332','1184649345','1164686879',
                '1164426896','1154302727','1144548322','1144266024','1144211301',
                '1114969169','1114924834','1104819366','1093894131','1093808040',
                '1083949382','1083789630','1083630073','1073053757','1063702785',
                '1053632463','1043447253','1033439732','1023188851','1023105400',
                '1013924372','1013924182','1013143213','1013062769','1003961251',
                '1003947599','1003878539','1003102781','1003063280'
            ) OR a.hco_npi_active IN (
                SELECT DISTINCT primary_npi
                FROM com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi
                WHERE secondary_npi IN ('1932280666','1912939703','1891765178','1861439952','1851458038',
                    '1831318856','1760480503','1760476659','1750482022','1750458485',
                    '1700128592','1689747552','1679973364','1669683512','1669462420',
                    '1669429577','1659877280','1649347469','1649261462','1639370059',
                    '1609824010','1598784555','1578693321','1568596765','1548212988',
                    '1477643690','1477549756','1467525790','1447423959','1437365186',
                    '1396882205','1376544320','1366556227','1366515488','1356496772',
                    '1346297843','1336495910','1336245828','1326092404','1295789907',
                    '1285832634','1285647933','1285174649','1275694184','1275564098',
                    '1265694442','1235582925','1235339227','1235234535','1235214834',
                    '1235148594','1225259039','1225249865','1215921457','1205935012',
                    '1205822236','1194787218','1184779332','1184649345','1164686879',
                    '1164426896','1154302727','1144548322','1144266024','1144211301',
                    '1114969169','1114924834','1104819366','1093894131','1093808040',
                    '1083949382','1083789630','1083630073','1073053757','1063702785',
                    '1053632463','1043447253','1033439732','1023188851','1023105400',
                    '1013924372','1013924182','1013143213','1013062769','1003961251',
                    '1003947599','1003878539','1003102781','1003063280'
                )
            ))
            THEN a.hco_npi_active
            ELSE '-'
        END AS hco_target
    FROM julie_file_mapping a
),

/* ---------------------------------------------------------------------------
   STEP 14: Fetch latest HCO ZIP from VOD
   - Pull latest VALID address (status A/DS, verified not NS/U) for HCO entity
   --------------------------------------------------------------------------- */
hco_zip_v1 AS (
    SELECT DISTINCT
        a.npi_num__v AS hco_npi_active,
        b.address_line_1__v as hco_address,
        b.postal_code_cda__v AS hco_postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_hco a
    JOIN com_raw.vod_address b
        ON b.entity_vid__v = a.vid__v
       AND b.entity_type__v = 'HCO'
       AND b.record_state__v = 'VALID'
       AND b.address_status__v IN ('A','DS')
       AND b.address_verification_status__v NOT IN ('NS','U')
    WHERE a.npi_num__v IN (
        SELECT DISTINCT hco_npi_active
        FROM updating_hco_target_flag
        WHERE hco_npi_active != '-'
    )
),

/* ---------------------------------------------------------------------------
   STEP 15: Select latest ZIP per HCO (rn=1)
   --------------------------------------------------------------------------- */
hco_zip_v2 AS (
    SELECT
        hco_npi_active,
        hco_address,
        hco_postal_code
    FROM hco_zip_v1
    WHERE rn = 1
),

/* ---------------------------------------------------------------------------
   STEP 16: Enrich / backfill HCO ZIP using:
   1) VOD latest ZIP (preferred)
   2) Komodo provider_zip (fallback)
   --------------------------------------------------------------------------- */
pulling_hco_zip_using_vod_komodo AS (
    SELECT
        a.* EXCEPT (hco_zip),
        case when b.hco_postal_code is not null then b.hco_address else c.provider_address end as hco_address,
        COALESCE(b.hco_postal_code, c.provider_zip, '-') AS hco_zip
    FROM updating_hco_target_flag a
    LEFT JOIN hco_zip_v2 b
        ON a.hco_npi_active = b.hco_npi_active
    LEFT JOIN com_raw.kom_providers c
        ON a.hco_npi_active = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 17: Reassign territory & region using ZIP
   - Uses HCO ZIP if present; otherwise falls back to HCP ZIP
   --------------------------------------------------------------------------- */
territory_region_reassignment AS (
    SELECT
        a.* EXCEPT (territory, region),
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region
    FROM pulling_hco_zip_using_vod_komodo a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON COALESCE(
               TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT),
               TRY_CAST(NULLIF(a.hcp_zipcode, '-') AS BIGINT)
           ) = b.zipcode
),

/* ---------------------------------------------------------------------------
   STEP 18: Final structural cleanup / de-dup
   --------------------------------------------------------------------------- */
final_output_v1 AS (
    SELECT DISTINCT
        hcp_npi,
        hcp_target,
        hcp_first_name,
        hcp_last_name,
        hcp_zipcode AS hcp_zip,
        hco_npi,
        hco_npi_active,
        hco_npi_present_julies_file_flag,
        hco_target,
        hco_name,
        hco_name_active,
        hco_address,
        hco_zip,
        territory_id,
        territory,
        region_id,
        region
    FROM territory_region_reassignment
),

/* ---------------------------------------------------------------------------
   STEP 19: Final enrichment with HCP specialty (Komodo INDIVIDUAL) + formatting
   --------------------------------------------------------------------------- */
final_output_v2 AS (
    SELECT DISTINCT
        hcp_npi,
        hcp_target,
        hcp_first_name,
        hcp_last_name,
        CONCAT(hcp_first_name, ' ', hcp_last_name) AS hcp_name,
        COALESCE(primary_specialty, '-') AS hcp_specialty,
        COALESCE(secondary_specialty, '-') AS hcp_secondary_specialty,
        COALESCE(hcp_zip, '-') AS hcp_zip,
        hco_npi as hco_npi_old,
        hco_name as hco_name_old,
        hco_npi_active AS hco_npi,
        hco_npi_present_julies_file_flag,
        hco_target,
        hco_name_active AS hco_name,
        coalesce(hco_address, '-') as hco_address,
        hco_zip,
        COALESCE(CAST(TRY_CAST(territory_id AS BIGINT) AS STRING), '-') AS territory_id,
        COALESCE(territory, '-') AS territory,
        COALESCE(CAST(TRY_CAST(region_id AS BIGINT) AS STRING), '-') AS region_id,
        COALESCE(region, '-') AS region
    FROM final_output_v1
    LEFT JOIN com_raw.kom_providers
        ON hcp_npi = npi
       AND provider_type = 'INDIVIDUAL'
)

SELECT *
FROM final_output_v2;
